# Sleep Monitor Analysis
**Data:** 4 combined `.mat` files — 12 sessions (6 subjects × 2 nights)

| File | Variable | Channels |
|------|----------|----------|
| `combinedCap.mat` | `dataCellCap` | CH, CLE, CRE, ax, ay, az |
| `combinedOther.mat` | `dataCellOther` | EEG, EOGl, EOGr, ECG, flow, pleth, thorax, abdomen |
| `combinedSP.mat` | `spCell` | sleep stage annotations |
| `combinedTime.mat` | `timeCell` | timestamps |

## Section 1 — Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, spectrogram, welch
from scipy.interpolate import interp1d
from scipy.ndimage import uniform_filter1d
import pandas as pd
from pathlib import Path

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 9,
})

In [ ]:
DATA_DIR = Path(r'C:\Users\adity\Documents\sleep monitor\allSubjects')

# Session metadata: index -> {subject, night, label}
SESSION_META = {
    i: {
        'subject': f'OS{(i // 2) + 1:03d}',
        'night': (i % 2) + 1,
        'label': f'S{(i // 2) + 1}N{(i % 2) + 1}'
    }
    for i in range(12)
}

CAP_CHANNELS   = ['CH', 'CLE', 'CRE', 'ax', 'ay', 'az']
OTHER_CHANNELS = ['EEG', 'EOGl', 'EOGr', 'ECG', 'flow', 'pleth', 'thorax', 'abdomen']

# Sleep stage integers (MATLAB convention from OS_PlotAllTogether.m numMat)
# Wake=4, N1=3, N2=2, N3=1, REM=0
STAGE_LABELS = {0: 'REM', 1: 'N3', 2: 'N2', 3: 'N1', 4: 'Wake'}
STAGE_COLORS = {0: '#9B59B6', 1: '#2ECC71', 2: '#3498DB', 3: '#F39C12', 4: '#E74C3C'}

EEG_BANDS = {
    'delta': (0.5,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0, 13.0),
    'beta' : (13.0, 30.0),
}

## Section 2 — Data Loading

In [ ]:
def unpack_mat_cell_2d(raw_cell):
    """
    Unpack a (1,12) MATLAB cell array where each element is a 2D numeric matrix.
    Returns a list of 12 numpy arrays, each shape (N_i, C).

    MATLAB cell arrays loaded by scipy nest as:
      raw_cell[0, i]          -> shape (1,1) object
      raw_cell[0, i][0, 0]   -> shape (N, C) float64
    """
    sessions = []
    for elem in np.asarray(raw_cell).ravel():
        arr = np.asarray(elem)
        # Unwrap (1,1) object wrapper — stop as soon as array is numeric
        while arr.dtype == object and arr.size == 1:
            arr = np.asarray(arr.flat[0])
        if arr.ndim == 1:
            arr = arr[:, np.newaxis]
        sessions.append(arr.astype(np.float64))
    return sessions


def unpack_mat_cell_objects(raw_cell):
    """
    Unpack a (1,12) MATLAB cell array where contents may not be numeric
    (e.g. datetime objects, string arrays).
    Returns a list of 12 raw arrays for manual inspection/parsing.
    """
    sessions = []
    for elem in np.asarray(raw_cell).ravel():
        arr = np.asarray(elem)
        while arr.dtype == object and arr.shape == (1, 1):
            arr = np.asarray(arr[0, 0])
        sessions.append(arr)
    return sessions

In [ ]:
# Load all .mat files
# Note: combinedOther.mat is ~463 MB — this cell may take 10–30 seconds
raw = {}
for fname in ['combinedCap.mat', 'combinedOther.mat', 'combinedSP.mat', 'combinedTime.mat']:
    print(f'Loading {fname}...', end=' ')
    raw[fname] = loadmat(DATA_DIR / fname)
    var_keys = [k for k in raw[fname] if not k.startswith('__')]
    print(f'done  (variables: {var_keys})')

In [ ]:
cap_sessions   = unpack_mat_cell_2d(raw['combinedCap.mat']['dataCellCap'])
other_sessions = unpack_mat_cell_2d(raw['combinedOther.mat']['dataCellOther'])
time_sessions  = unpack_mat_cell_objects(raw['combinedTime.mat']['timeCell'])
sp_sessions    = unpack_mat_cell_objects(raw['combinedSP.mat']['spCell'])

print(f"{'Sess':>5}  {'Label':>6}  {'cap shape':>12}  {'other shape':>12}")
print('-' * 45)
for i in range(12):
    print(f"{i:>5}  {SESSION_META[i]['label']:>6}  "
          f"{str(cap_sessions[i].shape):>12}  {str(other_sessions[i].shape):>12}")

print('\n--- Inspect time and sp dtypes (session 0) ---')
print(f'time_sessions[0] dtype: {time_sessions[0].dtype},  shape: {time_sessions[0].shape}')
print(f'sp_sessions[0]   dtype: {sp_sessions[0].dtype},    shape: {sp_sessions[0].shape}')
print(f'time sample [0:3]: {time_sessions[0].ravel()[:3]}')
print(f'sp   sample [0:3]: {sp_sessions[0].ravel()[:3]}')

In [ ]:
# ---------------------------------------------------------------------------
# Sleep stage extraction from combinedSP.mat workspace bytes
# ---------------------------------------------------------------------------
# spCell is a MATLAB MCOS 'table' object — scipy cannot decode it.
# Stage strings (e.g. '000; Wake', '000; Stage 2') survive as ASCII inside
# the raw __function_workspace__ blob and are recoverable via regex.

import re
from collections import Counter

_SP_STAGE_PATTERN = re.compile(
    rb'[\x20-\x7e]{1,25}; (Wake|Stage 1|Stage 2|Stage 3|Rem|Artefact|Artifact)',
    re.IGNORECASE
)
_SP_LABEL_MAP = {
    b'wake': 4, b'stage 1': 3, b'stage 2': 2, b'stage 3': 1,
    b'rem':  0, b'artefact': 4, b'artifact': 4,
}


def load_sleep_stages_from_sp(sp_mat_path, cap_sessions, epoch_sec=30.0, fs=100.0):
    """
    Extract sleep stages by scanning combinedSP.mat raw bytes.

    spCell is an undecodable MCOS table, but stage label strings survive as
    ASCII in the __function_workspace__ blob.

    Returns list of 12 int arrays: Wake=4, N1=3, N2=2, N3=1, REM=0, -1=unknown.
    """
    with open(sp_mat_path, 'rb') as f:
        workspace = f.read()

    matches = [m.group(1).lower() for m in _SP_STAGE_PATTERN.finditer(workspace)]
    epoch_counts = [int(np.ceil(cap_sessions[i].shape[0] / (epoch_sec * fs)))
                    for i in range(12)]
    print(f'Found {len(matches)} stage labels; '
          f'expected ~{sum(epoch_counts)} epochs across 12 sessions')

    sessions, offset = [], 0
    for n_ep in epoch_counts:
        chunk = matches[offset:offset + n_ep]
        codes = np.array([_SP_LABEL_MAP.get(lbl, -1) for lbl in chunk], dtype=int)
        if len(codes) < n_ep:
            codes = np.concatenate([codes, np.full(n_ep - len(codes), -1, dtype=int)])
        sessions.append(codes)
        offset += n_ep
    return sessions


sp_stage_sessions = load_sleep_stages_from_sp(DATA_DIR / 'combinedSP.mat', cap_sessions)

print(f'\n{"Sess":>5}  {"Label":>6}  {"#ep":>5}  Stage distribution')
print('-' * 60)
for i, codes in enumerate(sp_stage_sessions):
    c = Counter(codes.tolist())
    dist = '  '.join(f'{STAGE_LABELS[k]}={c[k]}' for k in sorted(c) if k in STAGE_LABELS)
    print(f'{i:>5}  {SESSION_META[i]["label"]:>6}  {len(codes):>5}  {dist}')


## Section 3 — Helper Functions

### Known scipy limitation: MATLAB MCOS objects

`timeCell` and `spCell` are MATLAB MCOS objects (class `datetime` and `table` respectively).
`scipy.io.loadmat` cannot decode them and returns an opaque structured array.

**Workaround applied here:**
- **Time** → use `dataCellCap` column 0 (`timeMS`, milliseconds from start, step=10 ms → Fs=100 Hz)
- **Sleep stages** → `parse_sleep_stages()` detects the opaque object and returns `-1` (unknown) epochs

**To get real sleep stages**, re-save `spCell` from MATLAB as a plain numeric matrix before saving:
```matlab
% In MATLAB, convert the table to a numeric array then save
spNum = {};
for i = 1:12
    tbl = spCell{i};
    spNum{i} = tbl{:, 2};   % adjust column index as needed
end
save('combinedSP_num.mat', 'spNum')
```
Then load `combinedSP_num.mat` and pass it to `sp_sessions` in Cell 2.3.

In [ ]:
# ---- Time helpers ----

def get_time_hours(session_idx):
    """Return time vector for session as hours elapsed from start."""
    t_ms = cap_sessions[session_idx][:, 0]   # timeMS column
    return (t_ms - t_ms[0]) / 3_600_000.0    # ms → hours


def compute_fs(session_idx):
    """Sampling frequency (Hz) from the timeMS column."""
    t_ms  = cap_sessions[session_idx][:, 0]
    dt_ms = np.median(np.diff(t_ms))
    return 1000.0 / dt_ms   # typically 100.0 Hz


# ---- Sleep stage helper ----

def parse_sleep_stages(session_idx):
    """
    Return (t_ep_hr, stage_ints) for session.
    Stage encoding: Wake=4, N1=3, N2=2, N3=1, REM=0, -1=unknown.
    Uses stages extracted from combinedSP.mat workspace bytes (cell 2.4).
    """
    t_hr  = get_time_hours(session_idx)
    codes = sp_stage_sessions[session_idx]
    t_ep  = np.linspace(t_hr[0], t_hr[-1], len(codes))
    return t_ep, codes


# ---- Channel accessors ----

def get_cap(session_idx):
    """Return (N, 6) cap matrix: [CH, CLE, CRE, ax, ay, az]."""
    return cap_sessions[session_idx][:, 1:7]   # skip col 0 = timeMS


def get_other(session_idx):
    """Return (N, 8) matrix: [EEG, EOGl, EOGr, ECG, flow, pleth, thorax, abdomen]."""
    return other_sessions[session_idx]


def acc_magnitude(cap_matrix):
    """Acceleration vector magnitude from ax, ay, az (columns 3, 4, 5 of cap matrix)."""
    ax, ay, az = cap_matrix[:, 3], cap_matrix[:, 4], cap_matrix[:, 5]
    return np.sqrt(ax**2 + ay**2 + az**2)


## Section 4 — Exploratory Overview

In [ ]:
rows = []
for i in range(12):
    t_hr = get_time_hours(i)
    fs   = compute_fs(i)
    t_ep, stages = parse_sleep_stages(i)
    rows.append({
        'Session': i,
        'Label':   SESSION_META[i]['label'],
        'Subject': SESSION_META[i]['subject'],
        'Night':   SESSION_META[i]['night'],
        'Cap shape':   cap_sessions[i].shape,
        'Other shape': other_sessions[i].shape,
        'Duration (hr)': round(t_hr[-1] - t_hr[0], 2),
        'Fs (Hz)':   round(fs, 1),
        'SP epochs': len(stages),
    })

df_overview = pd.DataFrame(rows).set_index('Session')
display(df_overview)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9), sharey=False)
stage_order = [k for k in sorted(STAGE_LABELS) if k >= 0]

for i, ax in enumerate(axes.flat):
    _, stages = parse_sleep_stages(i)
    counts = [np.sum(stages == s) for s in stage_order]
    colors = [STAGE_COLORS[s] for s in stage_order]
    ax.bar([STAGE_LABELS[s] for s in stage_order], counts, color=colors)
    ax.set_title(SESSION_META[i]['label'], fontsize=9)
    ax.tick_params(axis='x', rotation=40, labelsize=7)
    ax.set_ylabel('Epochs', fontsize=7)

fig.suptitle('Sleep Stage Epoch Counts per Session', fontsize=11)
plt.tight_layout()
plt.show()

## Section 5 — Single-Session Time Series

In [ ]:
SESSION = 0  # Change to inspect a different session (0–11)

t_hr        = get_time_hours(SESSION)
cap         = get_cap(SESSION)
other       = get_other(SESSION)
t_ep, stages = parse_sleep_stages(SESSION)

# Baseline-correct capacitive channels
CH  = cap[:, 0] - cap[0, 0]
CLE = cap[:, 1] - cap[0, 1]
CRE = cap[:, 2] - cap[0, 2]
aMag = acc_magnitude(cap)

EEG    = other[:, 0]
thorax = other[:, 6]
abdomen = other[:, 7]

channels = [
    ('CH (cap)',      t_hr, CH,      '#2980B9'),
    ('CLE (cap)',     t_hr, CLE,     '#27AE60'),
    ('CRE (cap)',     t_hr, CRE,     '#8E44AD'),
    ('Acc magnitude', t_hr, aMag,    '#E67E22'),
    ('EEG',          t_hr, EEG,     '#C0392B'),
    ('Thorax',       t_hr, thorax,  '#16A085'),
    ('Abdomen',      t_hr, abdomen, '#2C3E50'),
]

fig, axes = plt.subplots(len(channels), 1, figsize=(14, 16), sharex=True)

for ax, (label, t, sig, color) in zip(axes, channels):
    ax.plot(t, sig, color=color, lw=0.5, alpha=0.9)
    ax.set_ylabel(label, fontsize=8)

# Overlay hypnogram on top panel
ax_top = axes[0].twinx()
ax_top.step(t_ep, stages, color='#E74C3C', lw=0.8, where='post', alpha=0.7)
ax_top.set_ylim(-0.5, 4.5)
ax_top.set_yticks(list(STAGE_LABELS.keys()))
ax_top.set_yticklabels([STAGE_LABELS[k] for k in sorted(STAGE_LABELS)], fontsize=6)

axes[-1].set_xlabel('Time (hours from start)')
fig.suptitle(f"Session {SESSION_META[SESSION]['label']} — Full Night Overview", fontsize=11)
plt.tight_layout()
plt.show()

## Section 6 — Hypnogram

In [ ]:
def plot_hypnogram(session_idx, ax=None, title=None):
    """Color-filled hypnogram for one session."""
    t_ep, stages = parse_sleep_stages(session_idx)
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 2.5))

    for k in range(len(stages) - 1):
        ax.axvspan(t_ep[k], t_ep[k + 1],
                   color=STAGE_COLORS.get(stages[k], '#AAAAAA'), alpha=0.45)
    ax.step(t_ep, stages, color='k', lw=0.9, where='post')
    ax.set_yticks(list(STAGE_LABELS.keys()))
    ax.set_yticklabels([STAGE_LABELS[k] for k in sorted(STAGE_LABELS)], fontsize=7)
    ax.set_xlim(t_ep[0], t_ep[-1])
    ax.set_xlabel('Time (hours from start)')
    ax.set_title(title or SESSION_META[session_idx]['label'])
    return ax


fig, ax = plt.subplots(figsize=(14, 2.5))
plot_hypnogram(SESSION, ax=ax, title=f"Hypnogram — {SESSION_META[SESSION]['label']}")

legend_handles = [Patch(color=STAGE_COLORS[s], alpha=0.6, label=STAGE_LABELS[s])
                  for s in sorted(STAGE_LABELS)]
ax.legend(handles=legend_handles, loc='upper right', fontsize=7, ncol=5)
plt.tight_layout()
plt.show()

## Section 7 — Sleep Statistics

In [ ]:
stat_rows = []
for i in range(12):
    t_ep, stages = parse_sleep_stages(i)
    epoch_dur_hr = float(np.median(np.diff(t_ep))) if len(t_ep) > 1 else (30 / 3600)
    total = len(stages)
    row = {
        'Label':   SESSION_META[i]['label'],
        'Subject': SESSION_META[i]['subject'],
        'Night':   SESSION_META[i]['night'],
        'TST (hr)': round(np.sum(stages != 4) * epoch_dur_hr, 2),
        'WASO (hr)': round(np.sum(stages == 4) * epoch_dur_hr, 2),
    }
    for s_int, s_name in STAGE_LABELS.items():
        row[f'{s_name} (%)'] = round(100 * np.sum(stages == s_int) / total, 1) if total else 0
    stat_rows.append(row)

df_stats = pd.DataFrame(stat_rows)
display(df_stats)

In [ ]:
x = np.arange(12)
night_colors = ['#3498DB' if SESSION_META[i]['night'] == 1 else '#2ECC71' for i in range(12)]
labels_x     = [SESSION_META[i]['label'] for i in range(12)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# TST bar chart
axes[0].bar(x, df_stats['TST (hr)'], color=night_colors)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels_x, rotation=45)
axes[0].set_ylabel('Total Sleep Time (hr)')
axes[0].set_title('TST per Session  (blue=Night1, green=Night2)')

# Stacked stage proportion bar chart
bottom = np.zeros(12)
for s_int in sorted(STAGE_LABELS):
    col  = f"{STAGE_LABELS[s_int]} (%)"
    vals = df_stats[col].values
    axes[1].bar(x, vals, bottom=bottom, color=STAGE_COLORS[s_int],
                label=STAGE_LABELS[s_int], alpha=0.85)
    bottom += vals

axes[1].set_xticks(x)
axes[1].set_xticklabels(labels_x, rotation=45)
axes[1].set_ylabel('Proportion (%)')
axes[1].set_title('Sleep Stage Composition')
axes[1].legend(loc='upper right', fontsize=7)

plt.tight_layout()
plt.show()

## Section 8 — Signal Processing

In [ ]:
def butter_bandpass(data, f_low, f_high, fs, order=3):
    """Zero-phase Butterworth bandpass filter (mirrors MATLAB butterBand)."""
    nyq = fs / 2.0
    if f_high >= nyq:
        raise ValueError(f'f_high={f_high} Hz >= Nyquist={nyq:.1f} Hz')
    b, a = butter(order, [f_low / nyq, f_high / nyq], btype='band')
    return filtfilt(b, a, data.astype(np.float64))


def detrend_segment(data, win_sec=10.0, fs=100.0):
    """Segment-wise linear detrend + mean removal (mirrors MATLAB detrendMeanFil)."""
    w   = max(1, int(np.ceil(fs * win_sec)))
    x   = data.astype(np.float64)
    out = x.copy()
    idx = np.arange(w, dtype=np.float64)
    for s in range(0, len(x) - w + 1, w):
        seg     = x[s:s + w]
        seg_dt  = seg - np.polyval(np.polyfit(idx, seg, 1), idx)
        out[s:s + w] = seg_dt - seg_dt.mean()
    return out


def outlier_clip(data, win_size=1000, threshold=7.0):
    """Remove outlier spikes via local-mean residual, then linear interpolation."""
    x     = data.astype(np.float64)
    trend = uniform_filter1d(x, size=int(win_size))
    res   = x - trend
    bad   = np.abs(res) > threshold * np.std(res)
    clean = x.copy(); clean[bad] = np.nan
    nans  = np.isnan(clean)
    if nans.any() and (~nans).sum() >= 2:
        fi = interp1d(np.where(~nans)[0], clean[~nans], kind='linear', fill_value='extrapolate')
        clean[nans] = fi(np.where(nans)[0])
    return clean

In [ ]:
def compute_band_powers(eeg, fs, t_ep, stages):
    """
    Compute mean Welch PSD power in each EEG band per sleep stage epoch.
    Returns dict: band_name -> array of power values (length = number of epochs).
    """
    epoch_dur_sec = float(np.median(np.diff(t_ep))) * 3600.0
    spe = int(np.round(epoch_dur_sec * fs))   # samples per epoch
    n_epochs = len(stages)
    powers = {band: np.full(n_epochs, np.nan) for band in EEG_BANDS}

    for k in range(n_epochs):
        start = k * spe
        end   = start + spe
        if end > len(eeg):
            break
        seg = eeg[start:end]
        nperseg = min(256, len(seg))
        f_psd, psd = welch(seg, fs=fs, nperseg=nperseg)
        for band, (flo, fhi) in EEG_BANDS.items():
            mask = (f_psd >= flo) & (f_psd <= fhi)
            if mask.any():
                powers[band][k] = np.trapz(psd[mask], f_psd[mask])
    return powers


# Example: compute EEG band powers for SESSION
fs_val = compute_fs(SESSION)
eeg_sig = get_other(SESSION)[:, 0]   # EEG is column 0
t_ep_s, stages_s = parse_sleep_stages(SESSION)

band_powers = compute_band_powers(eeg_sig, fs_val, t_ep_s, stages_s)
for band, vals in band_powers.items():
    valid = vals[~np.isnan(vals)]
    print(f'{band:6s}: median power = {np.median(valid):.4f}  (n={len(valid)} epochs)')

In [ ]:
# Cap signal spectrogram (detrended CH, 0–5 Hz)
fs_val = compute_fs(SESSION)
ch_raw = get_cap(SESSION)[:, 0]
ch_det = detrend_segment(ch_raw, win_sec=10, fs=fs_val)

nperseg  = min(2048, len(ch_det))
noverlap = min(1800, nperseg - 1)
f_sg, t_sg, Sxx = spectrogram(ch_det, fs=fs_val, nperseg=nperseg,
                               noverlap=noverlap, nfft=4096)

f_max = 5.0
mask_f = f_sg <= f_max
log_S  = 20 * np.log10(np.abs(Sxx[mask_f, :]) + 1e-12)

t_hr_sg = t_sg / 3600.0  # spectrogram time is in seconds from start

fig, (ax_sg, ax_hyp) = plt.subplots(2, 1, figsize=(14, 5),
                                     gridspec_kw={'height_ratios': [3, 1]},
                                     sharex=True)
im = ax_sg.pcolormesh(t_hr_sg, f_sg[mask_f], log_S,
                      cmap='viridis', shading='gouraud', vmin=-50, vmax=50)
plt.colorbar(im, ax=ax_sg, label='Power (dB)')
ax_sg.set_ylabel('Frequency (Hz)')
ax_sg.set_title(f'CH Cap Spectrogram — {SESSION_META[SESSION]["label"]}')

plot_hypnogram(SESSION, ax=ax_hyp)
ax_hyp.set_xlabel('Time (hours from start)')

plt.tight_layout()
plt.show()

In [ ]:
# EEG filtered band time series + hypnogram
eeg_sig = get_other(SESSION)[:, 0]
t_hr    = get_time_hours(SESSION)

fig, axes = plt.subplots(len(EEG_BANDS) + 1, 1, figsize=(14, 12), sharex=True)

band_colors = {'delta': '#C0392B', 'theta': '#E67E22', 'alpha': '#27AE60', 'beta': '#2980B9'}

for ax, (band, (flo, fhi)) in zip(axes, EEG_BANDS.items()):
    try:
        filtered = butter_bandpass(eeg_sig, flo, fhi, fs_val, order=3)
        ax.plot(t_hr, filtered, color=band_colors[band], lw=0.4, alpha=0.9)
    except ValueError as e:
        ax.text(0.5, 0.5, str(e), transform=ax.transAxes, ha='center', color='red')
    ax.set_ylabel(f'{band}\n({flo}–{fhi} Hz)', fontsize=7)

plot_hypnogram(SESSION, ax=axes[-1])
axes[-1].set_xlabel('Time (hours from start)')

fig.suptitle(f'EEG Bands — {SESSION_META[SESSION]["label"]}', fontsize=11)
plt.tight_layout()
plt.show()

## Section 9 — Multi-Session Comparison

In [ ]:
# 3×4 tiled cap signal overview with sleep stage overlay
fig, axes = plt.subplots(3, 4, figsize=(20, 10), sharey=False)

for i, ax in enumerate(axes.flat):
    t_hr   = get_time_hours(i)
    cap    = get_cap(i)
    t_ep, stages = parse_sleep_stages(i)

    CH  = cap[:, 0] - cap[0, 0]
    CLE = cap[:, 1] - cap[0, 1]
    CRE = cap[:, 2] - cap[0, 2]

    ax.plot(t_hr, CH,  color='#2980B9', lw=0.35, alpha=0.8, label='CH')
    ax.plot(t_hr, CLE, color='#27AE60', lw=0.35, alpha=0.8, label='CLE')
    ax.plot(t_hr, CRE, color='#8E44AD', lw=0.35, alpha=0.8, label='CRE')

    ax.set_xlim(t_hr[0], t_hr[-1])
    ax.set_title(SESSION_META[i]['label'], fontsize=8)

    ax_r = ax.twinx()
    ax_r.step(t_ep, stages, color='#E74C3C', lw=0.7, where='post', alpha=0.6)
    ax_r.set_ylim(-0.5, 4.5)
    ax_r.set_yticks([])

    if i % 4 == 0:
        ax.set_ylabel('Cap signal', fontsize=7)
    if i >= 8:
        ax.set_xlabel('Time (hr)', fontsize=7)

axes[0, 0].legend(loc='upper right', fontsize=6)
fig.suptitle('All Sessions — Cap Signals (CH/CLE/CRE) with Sleep Stage Overlay', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# 6×2 hypnogram grid: subject × night
fig, axes = plt.subplots(6, 2, figsize=(16, 14))

for subj in range(6):
    for night in range(2):
        sess_idx = subj * 2 + night
        ax = axes[subj, night]
        plot_hypnogram(sess_idx, ax=ax)
        ax.set_xlabel('')

# Column headers
axes[0, 0].set_title('Night 1', fontsize=10, fontweight='bold')
axes[0, 1].set_title('Night 2', fontsize=10, fontweight='bold')

fig.suptitle('Hypnograms — All Subjects and Nights', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# EEG band power heatmap: 12 sessions × (band × sleep stage)
band_names  = list(EEG_BANDS.keys())
stage_items = [(s_int, STAGE_LABELS[s_int]) for s_int in sorted(STAGE_LABELS)]
col_labels  = [f'{b}\n{s_name}' for b in band_names for _, s_name in stage_items]
n_cols      = len(band_names) * len(stage_items)

power_matrix = np.full((12, n_cols), np.nan)

for i in range(12):
    eeg_i  = get_other(i)[:, 0]
    fs_i   = compute_fs(i)
    t_ep_i, stages_i = parse_sleep_stages(i)
    bp_i   = compute_band_powers(eeg_i, fs_i, t_ep_i, stages_i)

    col = 0
    for band in band_names:
        for s_int, _ in stage_items:
            mask = stages_i == s_int
            if mask.any():
                valid = bp_i[band][mask]
                valid = valid[~np.isnan(valid)]
                if len(valid) > 0:
                    power_matrix[i, col] = np.median(valid)
            col += 1

# Log-transform for better dynamic range
log_pm = np.log10(power_matrix + 1e-12)

fig, ax = plt.subplots(figsize=(18, 5))
im = ax.imshow(log_pm, aspect='auto', cmap='plasma',
               vmin=np.nanpercentile(log_pm, 5),
               vmax=np.nanpercentile(log_pm, 95))
ax.set_xticks(range(n_cols))
ax.set_xticklabels(col_labels, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(12))
ax.set_yticklabels([SESSION_META[i]['label'] for i in range(12)])
ax.set_title('Median EEG Band Power by Sleep Stage — All Sessions  (log10 µV²/Hz)')
plt.colorbar(im, ax=ax, label='log10 Power')
plt.tight_layout()
plt.show()